In [ ]:
"""
Run D — Physical model + U-space robust projection smoothing
=============================================================
Score target: beat 7.5 (geosteering notebook approach, no artifacts)

Key idea from 7.5 notebook:
  The PF/Beam prediction is smoothed in U = TVT + Z - anchor space.
  A robust polynomial is fitted in U vs normalized MD, then blended back.
  This removes drift artifacts caused by particle filter noise while
  preserving the overall trajectory shape.

Formula:
  anchor = last known TVT + last known Z
  U_i = TVT_pred_i + Z_i - anchor
  Fit: U_i ≈ P_d(s_i),  s_i = (MD_i - MD_last) / (MD_end - MD_last)
  Projected: TVT_proj_i = anchor + U_proj_i - Z_i
  Final: (1 - beta) * TVT_pred + beta * TVT_proj     (beta=0.75)

Pipeline:
  1. Physical model for visible wells (near-perfect, <0.01 ft RMSE)
  2. For hidden wells:
     a. 32-seed PF ensemble (fast)
     b. 14-config beam ensemble
     c. Blend: 0.85*PF + 0.15*Beam
     d. Robust U-space polynomial projection (degree 4, 4 robust iters)
     e. Final: 0.25 * blend + 0.75 * projection

Additional: per-well mini-validation on last 15% of known section
  to gate whether projection helps (only apply if it reduces val error).

Execution time: ~90s for 3 wells
"""

import os, glob, time, warnings
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

warnings.filterwarnings('ignore')

T0 = time.time()
def elapsed(): return f"[{time.time()-T0:6.1f}s]"
def log(msg):  print(f"{elapsed()} {msg}", flush=True)


def find_input_dir():
    for c in ['/kaggle/input/rogii-wellbore-geology-prediction',
              '/kaggle/input/competitions/rogii-wellbore-geology-prediction']:
        if os.path.isdir(c): return c
    hits = glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True)
    if hits: return os.path.dirname(hits[0])
    raise FileNotFoundError

INPUT_DIR = find_input_dir()
TRAIN_DIR = os.path.join(INPUT_DIR, 'train')
TEST_DIR  = os.path.join(INPUT_DIR, 'test')
log(f"INPUT_DIR={INPUT_DIR}")

_hw_files  = sorted(glob.glob(os.path.join(TEST_DIR, '*__horizontal_well.csv')))
TEST_WELLS = [os.path.basename(f).split('__')[0] for f in _hw_files]
log(f"Test wells ({len(TEST_WELLS)}): {TEST_WELLS}")

train_wids = set(
    os.path.basename(f).split('__')[0]
    for f in glob.glob(os.path.join(TRAIN_DIR, '*__horizontal_well.csv'))
)
log(f"Train wells: {len(train_wids)}")

sample = pd.read_csv(os.path.join(INPUT_DIR, 'sample_submission.csv'))
sample['well']    = sample['id'].str[:8]
sample['row_idx'] = sample['id'].str.rsplit('_', n=1).str[-1].astype(int)
log(f"Submission: {len(sample)} rows, {sample['well'].nunique()} wells")

FORM_COLS = ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']

# Projection hyperparams from 7.5 notebook
PROJ_DEGREE        = 5      # polynomial degree in U-space
PROJ_ROBUST_ITERS  = 6      # IRLS iterations for robustness
PROJ_ROBUST_C      = 1.8    # Huber-like threshold (in std units)
PROJ_BLEND_BETA    = 0.70   # weight of projected vs raw PF/Beam blend
PF_BEAM_BLEND      = 0.82   # PF weight in PF+Beam pre-blend
VAL_FRAC           = 0.18   # fraction of known section used for mini-validation
SMOOTH_WINDOW      = 7      # smoothing window for auxiliary denoising


def smooth_series(y, window=SMOOTH_WINDOW, poly=3):
    y = np.asarray(y, dtype=float)
    if len(y) < 5:
        return y
    window = int(min(window, len(y)))
    if window % 2 == 0:
        window -= 1
    if window < 5 or poly >= window:
        return y
    try:
        return savgol_filter(y, window_length=window, polyorder=min(poly, window - 1), mode='interp')
    except Exception:
        return y


def load_well(wid, split='train'):
    base = TRAIN_DIR if split == 'train' else TEST_DIR
    hw = pd.read_csv(os.path.join(base, f'{wid}__horizontal_well.csv'))
    tw = pd.read_csv(os.path.join(base, f'{wid}__typewell.csv'))
    return hw, tw

# ── Physical model: best formation contact ─────────────────────────────────────
def best_physical_pred(hw, tw):
    kn = hw[hw['TVT_input'].notna()].copy()
    if len(kn) < 5:
        last = float(kn['TVT_input'].iloc[-1]) if len(kn) > 0 else 0.0
        return np.full(len(hw), last), np.inf, 'none'
    tw_geo = tw.dropna(subset=['Geology']) if 'Geology' in tw.columns else pd.DataFrame()
    best_pred = None; best_rmse = np.inf; best_col = 'none'
    for col in FORM_COLS:
        if col not in hw.columns: continue
        hw_col = hw[col].ffill().bfill()
        if hw_col.isna().all(): continue
        kn_col = hw_col.iloc[kn.index].values
        if np.isnan(kn_col).mean() > 0.5: continue
        contact_tvt = np.nan
        if len(tw_geo) > 0 and 'Geology' in tw_geo.columns:
            gm = tw_geo[tw_geo['Geology'] == col]
            if len(gm) > 0: contact_tvt = float(gm['TVT'].min())
        if np.isnan(contact_tvt) and col in tw.columns:
            vals = tw[col].dropna()
            if len(vals) > 0: contact_tvt = float(vals.median())
        if np.isnan(contact_tvt): continue
        pred_kn = contact_tvt - (kn['Z'].values - kn_col)
        md_kn = kn['MD'].values.astype(float)
        md_ref = (md_kn - md_kn[0]) / max(md_kn[-1] - md_kn[0], 1.0)
        resid = kn['TVT_input'].values.astype(float) - pred_kn
        slope = 0.0
        if len(md_ref) >= 3 and np.ptp(md_ref) > 0:
            try:
                slope = float(np.polyfit(md_ref, resid, 1)[0])
            except Exception:
                slope = float(np.nanmedian(np.diff(resid) / np.maximum(np.diff(md_ref), 1e-6)))
        offset = float(np.nanmedian(resid - slope * md_ref))
        pred = contact_tvt - (hw['Z'].values - hw_col.values)
        pred = pred + offset + slope * ((hw['MD'].values.astype(float) - md_kn[0]) / max(md_kn[-1] - md_kn[0], 1.0))
        rmse = float(np.sqrt(np.nanmean((kn['TVT_input'].values - pred[kn.index])**2)))
        if rmse < best_rmse:
            best_rmse = rmse; best_pred = pred.astype(float); best_col = col
    if best_pred is None:
        last = float(kn['TVT_input'].iloc[-1])
        best_pred = np.where(hw['TVT_input'].notna(), hw['TVT_input'].values, last).astype(float)
    return best_pred.astype(float), best_rmse, best_col

# ── U-space robust polynomial projection ──────────────────────────────────────
def u_space_projection(hw_ref, tvt_pred, degree=PROJ_DEGREE,
                       robust_iters=PROJ_ROBUST_ITERS, robust_c=PROJ_ROBUST_C):
    """
    Project TVT trajectory in U = TVT + Z - anchor space.
    U should evolve smoothly along MD for a well-behaved geological path.
    Robust polynomial fit removes outlier PF noise.

    Returns projected TVT array (same length as hw_ref).
    """
    kn = hw_ref[hw_ref['TVT_input'].notna()]
    if len(kn) < 5:
        return tvt_pred.copy()

    anchor = float(kn['TVT_input'].iloc[-1]) + float(kn['Z'].iloc[-1])
    last_MD = float(kn['MD'].iloc[-1])
    end_MD  = float(hw_ref['MD'].iloc[-1])
    md_span = max(end_MD - last_MD, 1.0)

    # Build U values over prediction zone
    ev = hw_ref[hw_ref['TVT_input'].isna()]
    if len(ev) == 0:
        return tvt_pred.copy()

    s_ev = (ev['MD'].values - last_MD) / md_span    # normalised MD in [0, 1]
    z_ev = ev['Z'].values.astype(float)
    U_ev = tvt_pred[list(ev.index)] + z_ev - anchor  # U = TVT + Z - anchor

    # Design matrix for polynomial
    deg = min(max(2, degree), len(ev) - 1)
    deg = min(deg, 5)
    if deg < 2:
        return tvt_pred.copy()

    X = np.column_stack([s_ev**d for d in range(deg + 1)])

    # Robust IRLS fitting
    weights = np.ones(len(ev))
    coef = None
    for it in range(max(1, robust_iters)):
        W = np.diag(weights)
        XtW = X.T @ W
        try:
            coef = np.linalg.solve(XtW @ X + 1e-8 * np.eye(deg + 1), XtW @ U_ev)
        except np.linalg.LinAlgError:
            coef = np.linalg.lstsq(X, U_ev, rcond=None)[0]
        if it < robust_iters - 1:
            resid = U_ev - X @ coef
            sigma = max(np.std(resid), 1e-6)
            # Huber-like re-weighting
            r_norm = np.abs(resid) / (robust_c * sigma)
            weights = np.where(r_norm <= 1.0, 1.0, 1.0 / (r_norm + 1e-9))

    if coef is None:
        return tvt_pred.copy()

    U_proj = X @ coef
    tvt_proj = anchor + U_proj - z_ev

    # Build full-length projected array
    tvt_out = tvt_pred.copy()
    tvt_out[list(ev.index)] = tvt_proj
    pred_idx = np.flatnonzero(hw_ref['TVT_input'].isna().values)
    if len(pred_idx) >= 5:
        tvt_out[pred_idx] = smooth_series(tvt_out[pred_idx], window=min(7, len(pred_idx)), poly=3)
    return tvt_out

# ── PF ensemble (32-seed, fast) ────────────────────────────────────────────────
def run_pf_ensemble(hw, tw, n_seeds=32, n_particles=500, scale=5.0):
    tw_s   = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy()

    last     = kn.iloc[-1]
    last_tvt = float(last['TVT_input']); last_Z = float(last['Z']); last_MD = float(last['MD'])
    tw_at_k  = np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(kn['GR'].fillna(0).values - tw_at_k), 10., 60.))
    tail = kn.tail(30); dt=np.diff(tail['TVT_input'].values); dz=np.diff(tail['Z'].values); dm=np.diff(tail['MD'].values); m=dm>0
    ir = float(np.median((dt+dz)[m]/dm[m])) if m.sum()>=3 else 0.
    slope = 0.0
    if len(tail) >= 3:
        md_tail = tail['MD'].values.astype(float)
        tvt_tail = tail['TVT_input'].values.astype(float)
        if np.ptp(md_tail) > 0:
            try:
                slope = float(np.polyfit(md_tail, tvt_tail, 1)[0])
            except Exception:
                slope = float(np.nanmedian(np.diff(tvt_tail) / np.maximum(np.diff(md_tail), 1e-6)))

    gr_interp = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean())
    md_v = ev['MD'].values.astype(float); z_v = ev['Z'].values.astype(float)
    gr_v = gr_interp.values.astype(float)[list(ev.index)]

    MOM=0.998; VN=0.002; PN=0.005; RP=0.1; RR=0.001; N=n_particles

    # Stratigraphic position: pos = TVT + Z
    ls = last_tvt + last_Z
    preds = []; liks = []

    for seed in range(n_seeds):
        rng = np.random.default_rng(seed)
        pos  = ls + 2.0 * rng.standard_normal(N) + slope * 2.0
        rate = ir + 0.01 * rng.standard_normal(N) + 0.1 * slope
        w    = np.ones(N) / N
        res  = np.empty(len(ev)); prev_MD = last_MD; log_lik = 0.

        for i in range(len(ev)):
            dm_step = max(md_v[i] - prev_MD, 1.)
            rate = MOM*rate + VN*rng.standard_normal(N)
            pos  = pos + rate*dm_step + PN*rng.standard_normal(N)
            tvt_p = np.clip(pos-z_v[i], tw_tvt[0]-100, tw_tvt[-1]+100); pos = tvt_p+z_v[i]
            eg = np.interp(tvt_p, tw_tvt, tw_gr); d = (gr_v[i]-eg)/gs
            lk = np.maximum(np.exp(-0.5*np.minimum(d**2, 600.)), 1e-300)
            log_lik += np.log(max(float((w*lk).sum()), 1e-300))
            w = w*lk; ws=w.sum(); w = w/ws if ws>0 else np.ones(N)/N
            if 1./(w**2).sum() < 0.5*N:
                cum=np.cumsum(w); u0=rng.uniform(0,1./N)
                idx=np.clip(np.searchsorted(cum,u0+np.arange(N)/N),0,N-1)
                pos=pos[idx]+RP*rng.standard_normal(N); rate=rate[idx]+RR*rng.standard_normal(N); w=np.ones(N)/N
            res[i] = float(np.dot(w, pos-z_v[i])); prev_MD = md_v[i]

        out = hw['TVT_input'].values.astype(float).copy(); out[list(ev.index)] = res
        preds.append(out); liks.append(log_lik)

    liks = np.array(liks); weights = np.exp((liks-liks.max())/scale); weights /= weights.sum()
    base = (weights[:,None]*np.stack(preds,0)).sum(0)
    return smooth_series(base, window=min(7, len(base)), poly=3)

# ── Beam ensemble (14 configs) ──────────────────────────────────────────────────
BEAM_CONFIGS = [
    (10,20.,144.,2),(10,8.,64.,2),(8,35.,220.,1),(10,14.,90.,5),(20,4.,36.,3),
    (12,12.,100.,3),(15,25.,180.,2),(20,30.,200.,2),(15,10.,80.,4),(25,6.,50.,3),
    (10,40.,300.,1),(12,18.,120.,5),(30,8.,70.,2),(10,50.,400.,0),
]


def beam_search_single(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r):
    n=len(hgr); nt=len(tw_tvt)
    if n==0: return np.array([last_tvt])
    s=pd.Series(hgr,dtype='float32').interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
    if r>0: s=s.rolling(r*2+1,center=True,min_periods=1).mean()
    sgr=s.to_numpy(np.float32)
    if len(sgr) >= 5:
        window = min(7, len(sgr) if len(sgr)%2==1 else len(sgr)-1)
        if window >= 5:
            sgr = savgol_filter(sgr, window_length=window, polyorder=3, mode='interp')
    si=int(np.searchsorted(tw_tvt,last_tvt,'left')); si=min(max(si,0),nt-1)
    MOVES=np.array([-2,-1,0,1,2],dtype=np.int64); MC=mc*np.array([2.,1.,0.,1.,2.])
    bidx=np.full(bs,si,dtype=np.int64); bcost=np.full(bs,np.inf); bcost[0]=0.; bn=1
    result=np.zeros(n)
    for step in range(n):
        gv=sgr[step]
        ni=bidx[:bn,None]+MOVES[None,:]; ci=np.clip(ni,0,nt-1); valid=(ni>=0)&(ni<nt)
        gr_e=(gv-tw_gr[ci])**2/es
        tot=np.where(valid,bcost[:bn,None]+gr_e+MC[None,:],np.inf)
        ni_f=ni.flatten()[valid.flatten()]; tot_f=tot.flatten()[valid.flatten()]
        ord_=np.argsort(tot_f); ni_s=ni_f[ord_]; tot_s=tot_f[ord_]
        _,first=np.unique(ni_s,return_index=True); ni_u=ni_s[first]; tot_u=tot_s[first]
        kept=min(bs,len(ni_u)); top=np.argpartition(tot_u,min(kept-1,len(tot_u)-1))[:kept]
        top=top[np.argsort(tot_u[top])]
        bidx[:kept]=ni_u[top]; bcost[:kept]=tot_u[top]
        if kept<bs: bidx[kept:]=bidx[kept-1]; bcost[kept:]=np.inf
        bn=kept; result[step]=tw_tvt[bidx[0]]
    return result


def run_beam_14(hw, tw):
    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0: return hw['TVT_input'].values.astype(float).copy()
    last_tvt=float(kn.iloc[-1]['TVT_input'])
    tw_s=tw.sort_values('TVT'); tw_tvt=tw_s['TVT'].values.astype(float)
    tw_gr=tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)
    gr_all=hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean()).values.astype(float)
    hgr=gr_all[list(ev.index)]
    results=[beam_search_single(hgr,tw_tvt,tw_gr,last_tvt,bs,mc,es,r) for (bs,mc,es,r) in BEAM_CONFIGS]
    out=hw['TVT_input'].values.astype(float).copy()
    out[list(ev.index)]=np.stack(results,0).mean(0)
    return smooth_series(out, window=min(7, len(out)), poly=3)

# ── Per-well mini-validation: test whether projection helps ────────────────────
def gated_projection(hw_ref, tw_ref, tvt_blend, val_frac=VAL_FRAC):
    """
    Use last val_frac of the known section to decide if projection helps.
    Returns projected TVT if it reduces RMSE, else returns tvt_blend unchanged.
    """
    kn = hw_ref[hw_ref['TVT_input'].notna()]
    n_kn = len(kn)
    n_val = max(3, int(n_kn * val_frac))

    if n_kn < 20 or n_val >= n_kn:
        # Not enough data for validation — apply projection unconditionally
        tvt_proj = u_space_projection(hw_ref, tvt_blend)
        return (1 - PROJ_BLEND_BETA) * tvt_blend + PROJ_BLEND_BETA * tvt_proj

    # Create masked version: hide last n_val known rows
    hw_masked = hw_ref.copy()
    val_idx   = kn.index[-n_val:]
    hw_masked.loc[val_idx, 'TVT_input'] = np.nan

    # Run fast PF on masked well
    try:
        tvt_pf_masked = run_pf_ensemble(hw_masked, tw_ref, n_seeds=16, n_particles=300, scale=5.0)
    except Exception:
        tvt_pf_masked = tvt_blend.copy()

    # Blend with beam on masked well
    try:
        tvt_beam_masked = run_beam_14(hw_masked, tw_ref)
    except Exception:
        tvt_beam_masked = tvt_pf_masked.copy()

    tvt_blend_masked = PF_BEAM_BLEND * tvt_pf_masked + (1 - PF_BEAM_BLEND) * tvt_beam_masked

    # Project the masked blend
    best_rmse = np.inf
    best_beta = PROJ_BLEND_BETA
    for beta in [0.55, 0.70, 0.85]:
        tvt_proj_masked = u_space_projection(hw_masked, tvt_blend_masked)
        tvt_final_proj  = (1 - beta) * tvt_blend_masked + beta * tvt_proj_masked
        true_val  = hw_ref.loc[val_idx, 'TVT_input'].values.astype(float)
        pred_proj = tvt_final_proj[val_idx]
        rmse_proj = float(np.sqrt(np.mean((true_val - pred_proj)**2)))
        if rmse_proj < best_rmse:
            best_rmse = rmse_proj
            best_beta = beta

    raw_rmse = float(np.sqrt(np.mean((hw_ref.loc[val_idx, 'TVT_input'].values.astype(float) - tvt_blend_masked[val_idx])**2)))
    log(f"    Mini-val RMSE: raw={raw_rmse:.3f}  projected={best_rmse:.3f}")

    if best_rmse < float(np.sqrt(np.mean((hw_ref.loc[val_idx, 'TVT_input'].values.astype(float) - tvt_blend_masked[val_idx])**2))):
        # Projection helps — apply to full-data blend
        tvt_proj_full = u_space_projection(hw_ref, tvt_blend)
        result = (1 - best_beta) * tvt_blend + best_beta * tvt_proj_full
        log(f"    → Projection applied (beta={best_beta:.2f})")
    else:
        result = tvt_blend
        log(f"    → Projection skipped (no improvement)")

    return result

# ── MAIN ──────────────────────────────────────────────────────────────────────
rows = []
n_wells = len(TEST_WELLS)
times_per_well = []

for wi, wid in enumerate(TEST_WELLS):
    t_well = time.time()
    log(f"━━ Well {wi+1}/{n_wells}: {wid} ━━")

    hw_te, tw_te = load_well(wid, 'test')
    ev_count = int(hw_te['TVT_input'].isna().sum())
    log(f"  Rows: {len(hw_te)} total, {ev_count} to predict")

    tvt_final = None
    hw_tr = tw_tr = None

    # ── Visible well: physical model ───────────────────────────────────────────
    if wid in train_wids:
        try:
            hw_tr, tw_tr = load_well(wid, 'train')
            hw_work = hw_te.copy()
            hw_work['TVT_input'] = hw_tr['TVT_input'].values
            phys_pred, phys_rmse, best_col = best_physical_pred(hw_work, tw_tr)
            log(f"  Physical [{best_col}] RMSE_known={phys_rmse:.5f} ft")
            if phys_rmse < 2.5:
                tvt_final = phys_pred
                km = hw_tr['TVT_input'].notna().values
                tvt_final[km] = hw_tr['TVT_input'].values[km]
                log("  → Physical accepted")
        except Exception as e:
            log(f"  Physical failed: {e}")

    # ── Hidden / fallback: PF + Beam + U-space projection ──────────────────────
    if tvt_final is None:
        tw_ref = tw_tr if tw_tr is not None else tw_te
        hw_ref = hw_te
        if hw_tr is not None:
            hw_ref = hw_te.copy()
            hw_ref['TVT_input'] = hw_tr['TVT_input'].values

        last_known = float(hw_ref['TVT_input'].dropna().iloc[-1]) \
            if hw_ref['TVT_input'].notna().any() else 0.0

        # PF ensemble (32 seeds)
        t_pf = time.time()
        try:
            tvt_pf = run_pf_ensemble(hw_ref, tw_ref, n_seeds=32, n_particles=500, scale=5.0)
            log(f"  PF 32-seed OK  ({time.time()-t_pf:.1f}s)")
        except Exception as e:
            log(f"  PF failed: {e}")
            tvt_pf = hw_ref['TVT_input'].fillna(last_known).values.astype(float)

        # Beam ensemble (14 configs)
        t_beam = time.time()
        try:
            tvt_beam = run_beam_14(hw_ref, tw_ref)
            log(f"  Beam 14-config OK  ({time.time()-t_beam:.1f}s)")
        except Exception as e:
            log(f"  Beam failed: {e}")
            tvt_beam = tvt_pf.copy()

        # Pre-blend: 0.82 PF + 0.18 Beam
        tvt_blend = PF_BEAM_BLEND * tvt_pf + (1 - PF_BEAM_BLEND) * tvt_beam

        # Gated U-space projection with per-well mini-validation
        t_proj = time.time()
        try:
            tvt_final = gated_projection(hw_ref, tw_ref, tvt_blend, val_frac=VAL_FRAC)
            log(f"  Projection OK  ({time.time()-t_proj:.1f}s)")
        except Exception as e:
            log(f"  Projection failed: {e}")
            tvt_final = tvt_blend

        # Lock known rows to exact values
        if hw_tr is not None:
            km = hw_tr['TVT_input'].notna().values
            tvt_final[km] = hw_tr['TVT_input'].values[km]
        else:
            km = hw_te['TVT_input'].notna().values
            tvt_final[km] = hw_te['TVT_input'].values[km]

    ws = sample[sample['well'] == wid]
    for _, row in ws.iterrows():
        ridx = int(row['row_idx'])
        rows.append({'id': row['id'], 'tvt': float(tvt_final[ridx])})

    elapsed_well = time.time() - t_well
    times_per_well.append(elapsed_well)
    avg_t = np.mean(times_per_well)
    remaining = (n_wells - wi - 1) * avg_t
    log(f"  Done: {len(ws)} rows  |  Well: {elapsed_well:.1f}s  |  ETA: {remaining/60:.1f} min")

submission = pd.DataFrame(rows)
submission.to_csv('submission.csv', index=False)
log(f"\n✅ submission.csv: {len(submission)} rows")
log(f"   TVT: mean={submission['tvt'].mean():.2f}  std={submission['tvt'].std():.2f}")
log(f"   Total: {(time.time()-T0)/60:.1f} min")
print(submission.head(10))

[   0.0s] INPUT_DIR=/kaggle/input/competitions/rogii-wellbore-geology-prediction
[   0.0s] Test wells (3): ['000d7d20', '00bbac68', '00e12e8b']
[   0.0s] Train wells: 773
[   0.2s] Submission: 14151 rows, 3 wells
[   0.2s] ━━ Well 1/3: 000d7d20 ━━
[   0.2s]   Rows: 5278 total, 3836 to predict
[   0.3s]   Physical [none] RMSE_known=inf ft
[  12.3s]   PF 32-seed OK  (12.0s)
[  15.9s]   Beam 14-config OK  (3.6s)
[  25.0s]     Mini-val RMSE: raw=0.784  projected=0.488
[  25.2s]     → Projection applied (beta=0.75)
[  25.2s]   Projection OK  (9.3s)
[  25.4s]   Done: 3836 rows  |  Well: 25.2s  |  ETA: 0.8 min
[  25.4s] ━━ Well 2/3: 00bbac68 ━━
[  25.4s]   Rows: 7559 total, 6014 to predict
[  25.5s]   Physical [none] RMSE_known=inf ft
[  44.2s]   PF 32-seed OK  (18.7s)
[  49.8s]   Beam 14-config OK  (5.6s)
[  64.1s]     Mini-val RMSE: raw=1.159  projected=0.298
[  64.4s]     → Projection applied (beta=0.75)
[  64.4s]   Projection OK  (14.6s)
[  64.8s]   Done: 6014 rows  |  Well: 39.4s  |  ETA